# RQ1: Prompt sensitivity of bias injection

Compares detection macro-F1 (bias classes only) on the main dataset with the two datasets built using alternative injection prompts.

Requires the outputs of `bash_scripts/rq1_detection.sh` and `bash_scripts/prompt_sensitivity.sh`.

In [ ]:
import os

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

ORIGINAL_DIR = "../outputs/bias_detection"
ALT_DIR = "../outputs/prompt_sensitivity/bias_detection"

models = {
    "GPT-5.4-mini": "gpt-5.4-mini",
    "Llama 3.1 8B": "llama-3.1-8b",
    "Qwen3 8B": "qwen3-8b",
    "Gemma 4 31B": "gemma-4-31b",
}

BIAS_TYPES = ["anchoring", "authority", "framing", "herding", "recency", "overconfidence"]
mlb = MultiLabelBinarizer(classes=BIAS_TYPES)
mlb.fit([BIAS_TYPES])

In [ ]:
def parse_biases(val):
    if pd.isna(val) or val == "none" or val == "":
        return []
    return [b.strip() for b in str(val).split(",") if b.strip() in BIAS_TYPES]

def calc_macro_f1(path):
    if not os.path.exists(path):
        return "-"
    df = pd.read_csv(path)
    df = df[df["biases"] != "none"].copy()
    df["true_labels"] = df["biases"].apply(parse_biases)
    df["pred_labels"] = df["predicted_biases"].apply(parse_biases)
    
    y_true = mlb.transform(df["true_labels"].tolist())
    y_pred = mlb.transform(df["pred_labels"].tolist())
    
    return round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3)

results = []

for display_name, model_name in models.items():
    results.append({
        "Model": display_name,
        "Original": calc_macro_f1(f"{ORIGINAL_DIR}/{model_name}/dataset_predictions_{model_name}.csv"),
        "Alt. 1": calc_macro_f1(f"{ALT_DIR}/{model_name}_alt1/dataset_predictions_{model_name}_alt1.csv"),
        "Alt. 2": calc_macro_f1(f"{ALT_DIR}/{model_name}_alt2/dataset_predictions_{model_name}_alt2.csv"),
    })

results_df = pd.DataFrame(results)
results_df